In [1]:
import pandas as pd
import time
import csv
import re
comp=pd.read_csv('glycaninsilico.csv', header =None)
sortcomp = comp.sort_values([1]).reset_index(drop=True)

In [2]:
#testing block

print("Beofre sorting")
for i in range(3):
    print(comp[1][i])
print("After sorting")
for i in range(22):
    print(str(sortcomp[1][i]) + " mass <-, composition ->" + str(sortcomp[0][i]))


Beofre sorting
1149.601
1540.785
1931.97
After sorting
1149.601 mass <-, composition ->(0, 3, 2, 0, 0)
1323.69 mass <-, composition ->(1, 3, 2, 0, 0)
1353.701 mass <-, composition ->(0, 4, 2, 0, 0)
1394.727 mass <-, composition ->(0, 3, 3, 0, 0)
1497.78 mass <-, composition ->(2, 3, 2, 0, 0)
1510.775 mass <-, composition ->(0, 3, 2, 1, 0)
1527.79 mass <-, composition ->(1, 4, 2, 0, 0)
1540.785 mass <-, composition ->(0, 3, 2, 0, 1)
1557.801 mass <-, composition ->(0, 5, 2, 0, 0)
1568.817 mass <-, composition ->(1, 3, 3, 0, 0)
1598.827 mass <-, composition ->(0, 4, 3, 0, 0)
1639.854 mass <-, composition ->(0, 3, 4, 0, 0)
1671.869 mass <-, composition ->(3, 3, 2, 0, 0)
1684.864 mass <-, composition ->(1, 3, 2, 1, 0)
1701.879 mass <-, composition ->(2, 4, 2, 0, 0)
1714.875 mass <-, composition ->(1, 3, 2, 0, 1)
1714.875 mass <-, composition ->(0, 4, 2, 1, 0)
1731.89 mass <-, composition ->(1, 5, 2, 0, 0)
1742.906 mass <-, composition ->(2, 3, 3, 0, 0)
1744.885 mass <-, composition ->(0, 4

In [3]:
#modified from programmiz
def binarycompositionsearch(array, x, low, high, boundary):
    # Repeat until the pointers low and high meet each other
    while low <= high:
        mid = low + (high - low)//2
        #print(f"mid is {mid} now")
        #print(f"value:{array[mid]-x}")
        if abs((array[mid])-x)< boundary:
            return mid
        elif ((array[mid])-x)< -(boundary):
            #print("mid<x")
            low = mid + 1
            #print(f"now lower bound is {low} and it's {array[low]}")
        else:
            #print("mid>x")
            high = mid - 1
            #print(f"now higher bound is {high} and it's {array[high]}")
    return -1

array = sortcomp[1]
callcomp = sortcomp[0]

x = 1714
boundary = 2
result = binarycompositionsearch(array, x, 0, len(array)-1, boundary)


#Waiting: Extend search to find out all 
#the index has a mass difference less than defined BOUNDARY

if result != -1:
    print("Element is present at index " + str(result))
    #go upper
    upsearch = result+1
    lowsearch = result-1
    if abs((array[upsearch])-x)< boundary:
        print("Element is ALSO present upper at index " + str(upsearch) + ":" + str(sortcomp[1][upsearch]))
        upsearch = result+1
    else:
        print("Stop expanding searching upper element")
    if abs((array[lowsearch])-x)< boundary:
        print("Element is present lower at index " + str(lowsearch)+ ":"  + str(sortcomp[1][lowsearch]))
        lowsearch = result-1
    else:
        print("Stop expanding searching lower element")
else:
    print("Not found")

Element is present at index 15
Element is ALSO present upper at index 16:1714.875
Stop expanding searching lower element


In [4]:
print(f"{array[15]}+{callcomp[15]}")

1714.875+(1, 3, 2, 0, 1)


In [5]:
print(f"{array[16]}+{callcomp[16]}")

1714.875+(0, 4, 2, 1, 0)


In [6]:
#this blcok is designed for looping through the spectra
def compileresult(result,boundary):
    foundindex = []
    interpretedmz = None
    interpretedcomp = None
    upper = True
    lower = True
    #print(f"result is {result}")
    if result != -1:
        interpretedmz = array[result]
        interpretedcomp = callcomp[result]
        foundindex.append((interpretedmz, interpretedcomp))
        interpretedmz = None
        interpretedcomp = None
        upsearch = result+1
        escapelower = False
        if result > 0:
            lowsearch = result-1
        else:
            lowsearch = 0
            escapelower = True
        print(foundindex)
        while abs((array[upsearch])-array[result])< boundary:
            #print("upper test")
            #print(f"upsearch{upsearch}")
            interpretedmz = array[upsearch]
            interpretedcomp = callcomp[upsearch]
            foundindex.append((interpretedmz, interpretedcomp))
            #print(f"debug mass diff{(array[upsearch]-array[result])} and index {upsearch} vs {result}")
            upsearch = upsearch+1
            #print(f"now the index is {upsearch}")
        #print("Stop expanding searching upper element")
        if escapelower is True:
            print("Trying to search a mass lower than in silico minimum")
        while abs((array[lowsearch])-array[result])< boundary and not escapelower:
            #print("lower test")
            #print(f"lowsearch{lowsearch}")
            interpretedmz = array[lowsearch]
            interpretedcomp = callcomp[lowsearch]
            foundindex.append((interpretedmz, interpretedcomp))
            lowsearch = lowsearch-1
            #print(f"debug mass diff{(array[upsearch]-array[result])}")
            #print(f"now the index is {lowsearch}")
        #print("Stop expanding searching lower element")
        #print(f"Found index: {foundindex}")
        return foundindex
    else:
        #print("None")
        return foundindex

result = binarycompositionsearch(array, x, 0, len(array)-1, boundary)
compileresult(result,boundary)

[(1714.875, '(1, 3, 2, 0, 1)')]


[(1714.875, '(1, 3, 2, 0, 1)'), (1714.875, '(0, 4, 2, 1, 0)')]

In [28]:
#read the file
searchinput = pd.read_csv('zf_sPerMeNG_brain MS2 summary from 44218 at 20230307-153550.csv', sep = '\t')
searchinputindex = searchinput["in [H+]"]
#set boundary
boundary = 2
foundindextmp = []
for i in range(100):
    print(f"mz from spectra file {searchinputindex[i]}")
    tmp = int(searchinputindex[i])
    result = binarycompositionsearch(array, tmp, 0, len(array)-1, boundary)
    compileresult(result,boundary)
    print("-"*10)

mz from spectra file 1026.795138515625
None
----------
mz from spectra file 1118.80612484375
None
----------
mz from spectra file 1118.806368984375
None
----------
mz from spectra file 1046.7845183984375
None
----------
mz from spectra file 1046.7847625390625
None
----------
mz from spectra file 1100.7962371484375
None
----------
mz from spectra file 1118.806857265625
None
----------
mz from spectra file 1054.7900115625
None
----------
mz from spectra file 1118.8064910546875
None
----------
mz from spectra file 1078.75632015625
None
----------
mz from spectra file 1100.79635921875
None
----------
mz from spectra file 1178.8396941796875
None
----------
mz from spectra file 1100.7959930078125
None
----------
mz from spectra file 1118.80612484375
None
----------
mz from spectra file 1178.8384734765625
None
----------
mz from spectra file 1100.7959930078125
None
----------
mz from spectra file 1078.7568084375
None
----------
mz from spectra file 1138.833224453125
None
----------
mz from sp

In [7]:
#try to add the returned value into df
searchinput = pd.read_csv('zf_sPerMeNG_brain MS2 summary from 44218 at 20230307-153550.csv', sep = '\t')
searchinputindex = searchinput["in [H+]"]
#set boundary
boundary = 2
foundindextmp = []
for i in range(len(searchinput)):
    print(f"mz from spectra file {searchinputindex[i]}")
    tmp = int(searchinputindex[i])
    result = binarycompositionsearch(array, tmp, 0, len(array)-1, boundary)
    foundindextmp.append(compileresult(result,boundary))
searchinput['Predicted composition'] = foundindextmp

searchinput.head(100)

mz from spectra file 1026.795138515625
mz from spectra file 1118.80612484375
mz from spectra file 1118.806368984375
mz from spectra file 1046.7845183984375
mz from spectra file 1046.7847625390625
mz from spectra file 1100.7962371484375
mz from spectra file 1118.806857265625
mz from spectra file 1054.7900115625
mz from spectra file 1118.8064910546875
mz from spectra file 1078.75632015625
mz from spectra file 1100.79635921875
mz from spectra file 1178.8396941796875
mz from spectra file 1100.7959930078125
mz from spectra file 1118.80612484375
mz from spectra file 1178.8384734765625
mz from spectra file 1100.7959930078125
mz from spectra file 1078.7568084375
mz from spectra file 1138.833224453125
mz from spectra file 1118.8064910546875
mz from spectra file 1178.8399383203125
mz from spectra file 1138.833712734375
mz from spectra file 1078.7566863671875
mz from spectra file 1166.8501922265625
mz from spectra file 1138.833712734375
mz from spectra file 1178.8406707421875
mz from spectra file

[(2334.206, '(0, 4, 6, 0, 0)')]
mz from spectra file 3120.6170104296875
[(3118.579, '(3, 4, 4, 1, 1)')]
mz from spectra file 1001.6221038476564
mz from spectra file 2379.34762875
[(2379.216, '(2, 3, 4, 0, 1)')]
mz from spectra file 2363.301974453125
[(2362.201, '(0, 3, 4, 2, 0)')]
mz from spectra file 2278.357394375
[(2276.164, '(0, 3, 5, 0, 1)')]
mz from spectra file 2679.56198421875
mz from spectra file 1561.9058562890625
mz from spectra file 1609.8897430078125
mz from spectra file 2309.385958828125
[(2310.159, '(1, 4, 2, 0, 2)')]
mz from spectra file 3095.86862484375
[(3096.547, '(0, 9, 2, 2, 0)')]
mz from spectra file 1208.7522918359375
mz from spectra file 2378.426486171875
[(2379.216, '(2, 3, 4, 0, 1)')]
mz from spectra file 2410.365451015625
[(2409.227, '(0, 5, 4, 1, 0)')]
mz from spectra file 3102.836154140625
[(3103.579, '(1, 5, 6, 0, 1)')]
mz from spectra file 1245.7469207421875
mz from spectra file 3867.3271209375
[(3866.953, '(1, 4, 7, 2, 1)')]
mz from spectra file 3904.272

[(2374.2, '(0, 9, 2, 0, 0)')]
mz from spectra file 2859.527471855469
[(2858.453, '(1, 5, 5, 0, 1)')]
mz from spectra file 2318.175021328125
mz from spectra file 2802.509493984375
[(2802.416, '(5, 3, 2, 0, 2)')]
mz from spectra file 2418.27682796875
mz from spectra file 2140.17819515625
[(2140.089, '(1, 7, 2, 0, 0)')]
mz from spectra file 1382.84616390625
mz from spectra file 2595.3212615625
[(2594.332, '(2, 3, 5, 1, 0)')]
mz from spectra file 3092.566134609375
[(3090.548, '(1, 5, 3, 2, 1)')]
mz from spectra file 2573.3640197070317
[(2572.3, '(1, 6, 3, 0, 1)')]
mz from spectra file 2423.373724296875
[(2424.227, '(2, 4, 2, 2, 0)')]
mz from spectra file 1560.8123504296875
mz from spectra file 2404.26510921875
mz from spectra file 2695.4162658007813
[(2695.38, '(1, 3, 6, 0, 1)')]
mz from spectra file 3308.66156609375
[(3307.679, '(0, 7, 6, 1, 0)')]
mz from spectra file 2200.164035
mz from spectra file 2222.1718475
[(2222.143, '(1, 5, 4, 0, 0)')]
mz from spectra file 2554.380099453125
[(255

mz from spectra file 2394.2431365625
[(2392.212, '(0, 3, 4, 1, 1)')]
mz from spectra file 1394.72799984375
[(1394.727, '(0, 3, 3, 0, 0)')]
mz from spectra file 3241.737615898437
[(3240.626, '(3, 7, 2, 1, 1)')]
mz from spectra file 2426.258517421875
[(2424.227, '(2, 4, 2, 2, 0)')]
mz from spectra file 3569.833685234375
[(3567.769, '(3, 3, 2, 2, 3)')]
mz from spectra file 1937.9737273828125
[(1935.99, '(1, 6, 2, 0, 0)')]
mz from spectra file 2192.139865078125
[(2192.132, '(2, 4, 4, 0, 0)')]
mz from spectra file 4004.07853078125
[(4004.036, '(5, 6, 6, 0, 1)')]
mz from spectra file 1855.9317351953125
mz from spectra file 3787.958807304688
[(3788.884, '(3, 6, 2, 2, 2)')]
mz from spectra file 3799.950018242188
[(3799.9, '(5, 3, 3, 1, 3)')]
mz from spectra file 3626.8853209765625
[(3627.826, '(1, 7, 5, 1, 1)')]
mz from spectra file 1579.785372890625
mz from spectra file 1557.8050262109375
[(1557.801, '(0, 5, 2, 0, 0)')]
mz from spectra file 2031.0411101953125
[(2031.038, '(0, 3, 4, 0, 1)')]
m

[(2785.4, '(3, 3, 2, 2, 1)')]
mz from spectra file 2962.4931700976563
[(2961.505, '(3, 5, 4, 0, 1)')]
mz from spectra file 2445.225897148437
mz from spectra file 2831.429388359375
[(2832.426, '(2, 6, 2, 2, 0)')]
mz from spectra file 2139.1122771875
[(2140.089, '(1, 7, 2, 0, 0)')]
mz from spectra file 2263.171270683594
[(2263.133, '(0, 3, 2, 2, 1)')]
mz from spectra file 2858.4526427539063
[(2856.437, '(2, 3, 3, 2, 1)')]
mz from spectra file 2632.310275234375
[(2630.342, '(1, 7, 4, 0, 0)')]
mz from spectra file 3168.6327575
[(3167.584, '(1, 7, 3, 0, 2)')]
mz from spectra file 2368.2060271875
[(2366.221, '(3, 4, 4, 0, 0)')]
mz from spectra file 2423.246310390625
[(2424.227, '(2, 4, 2, 2, 0)')]
mz from spectra file 2263.172091640625
[(2263.133, '(0, 3, 2, 2, 1)')]
mz from spectra file 2208.130587734375
[(2207.132, '(4, 3, 2, 1, 0)')]
mz from spectra file 2018.045626796875
[(2018.043, '(1, 4, 4, 0, 0)')]
mz from spectra file 2947.5019591601563
[(2946.469, '(1, 4, 3, 0, 3)')]
mz from spectr

...

mz from spectra file 4365.22110890625
[(4363.194, '(5, 5, 4, 4, 0)')]
mz from spectra file 1922.959201015625
mz from spectra file 1713.904025234375
[(1714.875, '(1, 3, 2, 0, 1)')]
mz from spectra file 1723.813693203125
mz from spectra file 4587.359564140625
[(4586.288, '(4, 8, 3, 3, 1)')]
mz from spectra file 4641.345281914062
[(4640.347, '(4, 7, 7, 1, 1)')]
mz from spectra file 3871.002114765625
[(3872.941, '(5, 5, 2, 4, 0)')]
mz from spectra file 5951.97892140625
[(5949.974, '(2, 4, 10, 2, 4)')]
mz from spectra file 4147.079995625
[(4146.11, '(3, 6, 8, 0, 1)')]
mz from spectra file 3136.589572109375
[(3137.574, '(2, 6, 3, 0, 2)')]
mz from spectra file 2144.091924980469
mz from spectra file 1475.68796078125
mz from spectra file 1436.65475765625
mz from spectra file 4931.4755309375
[(4932.462, '(4, 7, 5, 1, 3)')]
mz from spectra file 5194.663275078125
[(5194.604, '(5, 8, 6, 0, 3)')]
mz from spectra file 1699.888888515625
mz from spectra file 4683.329901054687
[(4681.373, '(4, 6, 8, 1, 

mz from spectra file 2414.185275234375
[(2415.226, '(0, 8, 3, 0, 0)')]
mz from spectra file 2540.208956875
[(2540.31, '(4, 4, 4, 0, 0)')]
mz from spectra file 3149.602239921875
[(3150.605, '(0, 8, 6, 0, 0)')]
mz from spectra file 1672.7749969140625
[(1671.869, '(3, 3, 2, 0, 0)')]
mz from spectra file 2122.107394375
[(2123.074, '(0, 6, 2, 1, 0)')]
mz from spectra file 1922.9615203515625
mz from spectra file 2924.492404140625
[(2922.458, '(1, 7, 2, 0, 2)')]
mz from spectra file 2988.3328307421875
[(2987.521, '(5, 3, 3, 2, 0)')]
mz from spectra file 1867.1906798632813
mz from spectra file 2507.237033046875
[(2508.259, '(0, 3, 3, 2, 1)')]
mz from spectra file 2331.1630584375
mz from spectra file 2742.377142265625
[(2742.395, '(5, 3, 2, 2, 0)')]
mz from spectra file 1871.922091640625
[(1871.948, '(0, 3, 2, 2, 0)')]
mz from spectra file 1876.8760711328125
[(1875.969, '(3, 4, 2, 0, 0)')]
mz from spectra file 2950.36716
[(2950.489, '(1, 8, 3, 1, 0)')]
mz from spectra file 1697.827365078125
mz 

mz from spectra file 2317.13913265625
mz from spectra file 2944.4638732226563
[(2944.49, '(3, 3, 4, 0, 2)')]
mz from spectra file 2352.147677578125
[(2351.185, '(1, 3, 3, 0, 2)')]
mz from spectra file 2892.458956875
[(2892.447, '(0, 8, 2, 2, 0)')]
mz from spectra file 3017.401583828125
[(3015.527, '(1, 4, 5, 1, 1)')]
mz from spectra file 2711.318331875
[(2710.38, '(2, 3, 4, 2, 0)')]
mz from spectra file 2974.469454921875
[(2972.485, '(2, 3, 2, 3, 1)')]
mz from spectra file 2472.2587615625
[(2471.253, '(3, 5, 2, 0, 1)')]
mz from spectra file 2552.2832701953125
[(2553.306, '(2, 4, 4, 1, 0)')]
mz from spectra file 2530.2978240625
[(2531.274, '(1, 7, 2, 0, 1)')]
mz from spectra file 3022.452337578125
[(3021.526, '(1, 7, 4, 0, 1)')]
mz from spectra file 3150.5825866015625
[(3150.605, '(0, 8, 6, 0, 0)')]
mz from spectra file 2742.3732970507813
[(2742.395, '(5, 3, 2, 2, 0)')]
mz from spectra file 3016.400607265625
[(3015.527, '(1, 4, 5, 1, 1)')]
mz from spectra file 2949.413790859375
[(2950.4

[(3387.753, '(3, 3, 9, 0, 0)')]
mz from spectra file 2946.41842953125
[(2944.49, '(3, 3, 4, 0, 2)')]
mz from spectra file 3558.7763121875
[(3556.789, '(2, 7, 4, 1, 1)')]
mz from spectra file 3164.587225273437
[(3163.626, '(5, 5, 5, 0, 0)')]
mz from spectra file 3147.552462734375
[(3146.574, '(3, 3, 2, 3, 1)')]
mz from spectra file 3142.596896328125
[(3142.627, '(3, 3, 8, 0, 0)')]
mz from spectra file 3385.7186675
[(3384.679, '(0, 7, 3, 0, 3)')]
mz from spectra file 3348.700872734375
[(3348.706, '(1, 5, 7, 0, 1)')]
mz from spectra file 2312.184786953125
[(2310.159, '(1, 4, 2, 0, 2)')]
mz from spectra file 2726.346408046875
[(2727.395, '(3, 4, 4, 1, 0)')]
mz from spectra file 2732.30075375
[(2731.353, '(0, 5, 2, 0, 3)')]
mz from spectra file 2686.33053890625
[(2686.368, '(3, 5, 3, 1, 0)')]
mz from spectra file 3372.580783046875
[(3371.684, '(0, 9, 3, 1, 1)')]
mz from spectra file 3185.505560234375
[(3184.599, '(1, 9, 3, 0, 1)')]
mz from spectra file 2703.35690609375
[(2701.343, '(0, 5, 2

,entry no,MS1scan no,MS1Isolation mass,MS1monoIsomass,chargeState,in [H+],intensityN\/A,StructureN\/A,MS2 Scan no,peaklist,Predicted composition
0,1,309,513.901489,513.901489,2,1026.795139,ext from peak list,structure na,311,"MS2peaklist(dMass=(97.00896453857422, 98.33753...",[]
1,2,320,559.906982,559.906982,2,1118.806125,ext from peak list,structure na,322,"MS2peaklist(dMass=(91.90139770507812, 91.91895...",[]
2,3,349,559.907104,559.907104,2,1118.806369,ext from peak list,structure na,351,"MS2peaklist(dMass=(91.93927764892578, 94.68694...",[]
3,4,350,523.896179,523.896179,2,1046.784518,ext from peak list,structure na,353,"MS2peaklist(dMass=(97.06316375732422, 97.59021...",[]
4,5,374,523.896301,523.896301,2,1046.784763,ext from peak list,structure na,376,"MS2peaklist(dMass=(93.80081176757812, 98.26014...",[]
...,...,...,...,...,...,...,...,...,...,...,...
95,96,1031,820.837769,820.837769,2,1640.667697,ext from peak list,structure na,1038,"MS2peaklist(dMass=(90.97655487060547, 102.8438...","[(1639.854, (0, 3, 4, 0, 0))]"
96,97,1032,699.856140,699.856140,2,1398.704440,ext from peak list,structure na,1040,"MS2peaklist(dMass=(90.94359588623047, 90.97645...",[]
97,98,1032,684.859924,684.859924,2,1368.712009,ext from peak list,structure na,1041,"MS2peaklist(dMass=(90.97647857666016, 97.36291...",[]
98,99,1032,808.862976,808.862976,2,1616.718112,ext from peak list,structure na,1042,"MS2peaklist(dMass=(90.79341888427734, 90.97680...",[]


In [8]:
#save the file into another file and generate a setting file... does setting file needed?
timestamp = time.strftime("%Y%m%d-%H%M%S") #datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
extfilename= "zf_sPerMeNG_brain MS2 summary from 44218 at 20230307-153550 " + timestamp + " predictedcomp.csv"
searchinput.to_csv(extfilename, sep = '\t')
print("export finished") #Doesn't check the existence of file, should be modified in the future

export finished


In [9]:
#find the folder contains *predictcomp.csv
import fnmatch
import os

for file in os.listdir('.'):
    if fnmatch.fnmatch(file, '*predictedcomp.csv'):
        print(file)
#https://docs.python.org/3/library/fnmatch.html#module-fnmatch

zf_sPerMeNG_brain MS2 summary from 44218 at 20230307-153550 20230507-181703 predictedcomp.csv
zf_sPerMeNG_brain MS2 summary from 44218 at 20230307-153550 20230511-143124 predictedcomp.csv
